# Feature Engineering v2

Referans: Kaya & Kalay, IEEE Access 2025

**Hedef:** Her segment (durak_i → durak_i+1) için `travel_time_min` tahmini.

**Kararlar:**
- Zaman filtresi: 06:00–24:00
- GTFS eşleşmeyenler: çıkarılır
- `congestion_ratio` yoksa: 1.0 (serbest akış) ile doldurulur
- `deviation_from_schedule` direkt feature değil — önceki segmentin lag'ı olarak kullanılır (target leakage önleme)

In [ ]:
import os
import math
import sqlite3
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
COLLECTOR_DIR = os.path.join(PROJECT_ROOT, 'data_collector')
GTFS_DIR     = os.path.join(PROJECT_ROOT, 'data', 'bus-eshot-gtfs')
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'collected_data')
OUTPUT_CSV   = os.path.join(OUTPUT_DIR, 'route_502_features_v2.csv')

DB_CANDIDATES = [
    os.path.join(OUTPUT_DIR, 'eshot_buses.db'),
    os.path.join(COLLECTOR_DIR, 'collected_data', 'eshot_buses.db'),
    os.path.join(COLLECTOR_DIR, 'collected_data', 'route_502_realtime.db'),
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)

import sys
sys.path.insert(0, COLLECTOR_DIR)
from config import STOPS_DIR0, STOPS_DIR1

if DB_PATH:
    print(f'DB: {DB_PATH}  ({os.path.getsize(DB_PATH)/1024:.1f} KB)')
else:
    raise FileNotFoundError('DB bulunamadi. collected_data/eshot_buses.db dosyasini kopyala.')

## Adım 1 — Ham Segmentleri Oluştur

**Düzeltmeler (v1 → v2):**
- `seq == prev_seq` (GPS jitter): new_trip açmak yerine o event atlanır
- `seq > prev_seq` (sefer sonu): new_trip açılır

In [ ]:
def build_segments(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("""
        SELECT otobus_id, route_id, yon, stop_id, stop_seq, stop_name, timestamp
        FROM trip_events
        WHERE event_type = 'arrival'
        ORDER BY otobus_id, yon, timestamp
    """).fetchall()
    conn.close()
    print(f'Toplam arrival eventi: {len(rows)}')

    bus_events = defaultdict(list)
    for row in rows:
        key = (row['otobus_id'], row['yon'], row['route_id'])
        bus_events[key].append({
            'ts':        datetime.strptime(row['timestamp'], '%Y-%m-%d %H:%M:%S'),
            'stop_seq':  row['stop_seq'],
            'stop_id':   row['stop_id'],
            'stop_name': row['stop_name'],
        })

    all_segments = []
    for (bus_id, yon, route_id), events in bus_events.items():
        current_trip = []
        prev_seq = None
        prev_ts  = None

        for ev in events:
            seq = ev['stop_seq']
            ts  = ev['ts']

            if prev_seq is not None:
                gap = (ts - prev_ts).total_seconds()

                if gap > 1800 or seq > prev_seq:
                    # 30 dk boşluk veya seq arttı → yeni sefer
                    all_segments.extend(_extract_segments(current_trip, bus_id, yon, route_id))
                    current_trip = []
                    prev_seq = None
                    prev_ts  = None
                elif seq == prev_seq:
                    # GPS jitter: aynı durağa tekrar gelme, event atla
                    continue
                # seq < prev_seq: normal ilerleme

            current_trip.append(ev)
            prev_seq = seq
            prev_ts  = ts

        all_segments.extend(_extract_segments(current_trip, bus_id, yon, route_id))

    return all_segments


def _extract_segments(events, bus_id, yon, route_id):
    if len(events) < 2:
        return []
    segs = []
    trip_start_ts = events[0]['ts']

    for i in range(1, len(events)):
        prev = events[i - 1]
        curr = events[i]

        if curr['stop_seq'] != prev['stop_seq'] - 1:
            continue

        travel_sec = (curr['ts'] - prev['ts']).total_seconds()
        travel_min = travel_sec / 60.0

        if not (0.33 <= travel_min <= 15):
            continue

        # Zaman filtresi: 06:00–24:00
        if curr['ts'].hour < 6:
            continue

        segs.append({
            'bus_id':            bus_id,
            'route_id':          route_id,
            'yon':               yon,
            'date':              trip_start_ts.strftime('%Y-%m-%d'),
            'trip_start_time':   trip_start_ts.strftime('%H:%M:%S'),
            'arrival_timestamp': curr['ts'].strftime('%Y-%m-%d %H:%M:%S'),
            'from_stop_seq':     prev['stop_seq'],
            'to_stop_seq':       curr['stop_seq'],
            'from_stop_name':    prev['stop_name'],
            'to_stop_name':      curr['stop_name'],
            'travel_seconds':    round(travel_sec, 1),
            'travel_time_min':   round(travel_min, 3),
        })
    return segs


raw_segments = build_segments(DB_PATH)
df = pd.DataFrame(raw_segments)
print(f'Toplam segment: {len(df)}')
if len(df) > 0:
    print(f'Tarih aralığı: {df["date"].min()} – {df["date"].max()}')
    print(f'\nHedef (travel_time_min):')
    print(df['travel_time_min'].describe().round(3))
df.head()

## Adım 2 — Zamansal Featurelar

| Feature | Açıklama |
|---------|----------|
| `hour`, `day_of_week` | Ham sayılar (referans için) |
| `day_type` | 0=hafta içi, 1=hafta sonu |
| `time_block` | 0=sabah pik, 1=gündüz, 2=akşam pik, 3=gece |
| `hour_sin/cos` | Çembersel encoding — 23 ile 0 arası fark 1 saat |
| `dow_sin/cos` | Çembersel encoding — Pazar ile Pazartesi komşu |

In [ ]:
df['arrival_timestamp'] = pd.to_datetime(df['arrival_timestamp'])
df['hour']        = df['arrival_timestamp'].dt.hour
df['day_of_week'] = df['arrival_timestamp'].dt.dayofweek  # 0=Pazartesi
df['day_type']    = (df['day_of_week'] >= 5).astype(int)

def get_time_block(hour):
    if 6  <= hour < 10: return 0  # sabah pik
    if 10 <= hour < 17: return 1  # gündüz
    if 17 <= hour < 20: return 2  # akşam pik
    return 3                      # gece

df['time_block'] = df['hour'].apply(get_time_block)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

block_labels = {0: 'sabah_pik', 1: 'gündüz', 2: 'akşam_pik', 3: 'gece'}
print('Zaman bloğu:', df['time_block'].map(block_labels).value_counts().to_dict())
print('Hafta içi/sonu:', df['day_type'].value_counts().to_dict())

## Adım 3 — GTFS Planlanmış Süre

**Eşleştirme notu:** Segmentlerde seq azalır (from=N, to=N-1), GTFS'te artar.  
`segment.to_seq` ↔ `gtfs.from_seq`, `segment.from_seq` ↔ `gtfs.to_seq`

GTFS eşleşmesi bulunamayan satırlar çıkarılır (`inner join`).

In [ ]:
trips_gtfs = pd.read_csv(os.path.join(GTFS_DIR, 'trips.txt'))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, 'stop_times.txt'))

route502_ids = set(trips_gtfs[trips_gtfs['route_id'] == 502]['trip_id'])
st502 = stop_times[stop_times['trip_id'].isin(route502_ids)].copy()
print(f'Route 502 GTFS kayıt: {len(st502)}')

def time_to_sec(t):
    h, m, s = t.strip().split(':')
    return int(h)*3600 + int(m)*60 + int(s)

st502['arr_sec'] = st502['arrival_time'].apply(time_to_sec)
st502['dep_sec'] = st502['departure_time'].apply(time_to_sec)
st502 = st502.sort_values(['trip_id', 'stop_sequence'])

sched_records = []
for _, grp in st502.groupby('trip_id'):
    grp = grp.sort_values('stop_sequence')
    seqs = grp['stop_sequence'].values
    deps = grp['dep_sec'].values
    arrs = grp['arr_sec'].values
    for i in range(1, len(seqs)):
        secs = arrs[i] - deps[i-1]
        if secs > 0:
            sched_records.append({'gtfs_from': int(seqs[i-1]), 'gtfs_to': int(seqs[i]), 'sched_sec': secs})

sched_avg = (
    pd.DataFrame(sched_records)
    .groupby(['gtfs_from', 'gtfs_to'])['sched_sec'].mean()
    .reset_index()
)
sched_avg['scheduled_travel_min'] = (sched_avg['sched_sec'] / 60).round(3)

before = len(df)
df = df.merge(
    sched_avg[['gtfs_from', 'gtfs_to', 'scheduled_travel_min']],
    left_on=['to_stop_seq', 'from_stop_seq'],
    right_on=['gtfs_from', 'gtfs_to'],
    how='inner'
).drop(columns=['gtfs_from', 'gtfs_to'])

print(f'GTFS eşleşmesi: {len(df)}/{before} ({len(df)/before*100:.1f}% kaldı)')
print(f'Planlanmış süre (dk): ort={df["scheduled_travel_min"].mean():.2f}, std={df["scheduled_travel_min"].std():.2f}')

## Adım 4 — Lag Featurelar (Önceki Segment Bilgisi)

**Target leakage düzeltmesi:**  
`deviation = travel_time_min - scheduled_travel_min` hedeften türetildiği için direkt feature olamaz.  
Bunun yerine **önceki segmentin** değerleri lag feature olarak kullanılır — bu bilgi tahmin anında mevcuttur.

| Feature | Açıklama |
|---------|----------|
| `prev_travel_time_min` | Önceki durağa geçen gerçek süre — makalenin `elapsed_time` feature'ı |
| `prev_deviation` | Önceki segmentte plandan sapma (gecikmeler birikir) |

İlk segment (trip başı): önceki bilgi yok → 0.0 ile doldurulur (nötr varsayım).

In [ ]:
# Her trip içinde segmentleri from_stop_seq'e göre sırala (azalan = ilerleme yönü)
df = df.sort_values(
    ['bus_id', 'yon', 'date', 'trip_start_time', 'from_stop_seq'],
    ascending=[True, True, True, True, False]
).reset_index(drop=True)

trip_group = ['bus_id', 'yon', 'date', 'trip_start_time']

# Önceki segmentin gerçek süresi
df['prev_travel_time_min'] = (
    df.groupby(trip_group)['travel_time_min']
    .shift(1)
    .fillna(0.0)
)

# Önceki segmentin sapması (trip içinde hesaplanır, leakage yok)
df['_deviation'] = df['travel_time_min'] - df['scheduled_travel_min']
df['prev_deviation'] = (
    df.groupby(trip_group)['_deviation']
    .shift(1)
    .fillna(0.0)
)
df = df.drop(columns=['_deviation'])

first_seg = (df['prev_travel_time_min'] == 0).sum()
print(f'Toplam segment     : {len(df)}')
print(f'İlk segment (0.0)  : {first_seg} ({first_seg/len(df)*100:.1f}%)')
print(f'\nprev_travel_time_min: ort={df["prev_travel_time_min"].mean():.2f} dk')
print(f'prev_deviation      : ort={df["prev_deviation"].mean():.2f} dk')

## Adım 5 — Hava Durumu Featureları

Arrival timestamp'e en yakın hava kaydıyla eşleştirilir (`merge_asof`, 2 saat tolerans).

In [ ]:
conn = sqlite3.connect(DB_PATH)
weather_df = pd.read_sql_query('SELECT * FROM weather_readings ORDER BY timestamp', conn)
conn.close()
print(f'Hava kayıt sayısı: {len(weather_df)}')

WEATHER_CAT_MAP = {'clear': 0, 'cloudy': 1, 'rainy': 2, 'snowy': 3}

if len(weather_df) > 0:
    weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
    weather_df['weather_cat_enc'] = weather_df['weather_category'].map(WEATHER_CAT_MAP).fillna(1)

    df = pd.merge_asof(
        df.sort_values('arrival_timestamp'),
        weather_df[['timestamp', 'temperature', 'humidity', 'precipitation',
                     'wind_speed', 'visibility', 'weather_category', 'weather_cat_enc']]
              .sort_values('timestamp'),
        left_on='arrival_timestamp',
        right_on='timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('2h')
    ).drop(columns=['timestamp'], errors='ignore')

    matched = df['temperature'].notna().sum()
    print(f'Eşleşme: {matched}/{len(df)} ({matched/len(df)*100:.1f}%)')
    print(f'Kaynak: {weather_df["source"].value_counts().to_dict()}')
else:
    print('Hava verisi henüz yok — NaN olarak eklendi')
    for col in ['temperature', 'humidity', 'precipitation', 'wind_speed',
                'visibility', 'weather_category', 'weather_cat_enc']:
        df[col] = np.nan

## Adım 6 — Trafik Featureları (TomTom)

**Düzeltme:** O(n²) nested loop yerine `merge_asof` + `by` parametresiyle O(n log n) merge.

Eşleşme bulunamazsa `congestion_ratio = 1.0` (serbest akış).

In [ ]:
conn = sqlite3.connect(DB_PATH)
traffic_df = pd.read_sql_query("""
    SELECT timestamp, from_stop_seq, to_stop_seq, direction, congestion_ratio
    FROM traffic_readings ORDER BY timestamp
""", conn)
conn.close()
print(f'Trafik kayıt sayısı: {len(traffic_df)}')

if len(traffic_df) > 0:
    traffic_df['timestamp'] = pd.to_datetime(traffic_df['timestamp'])

    # merge_asof: from/to/direction'da exact match, timestamp'te nearest
    # 'yon' → 'direction' eşleştirmesi için geçici kolon
    df['direction'] = df['yon']

    df = pd.merge_asof(
        df.sort_values('arrival_timestamp'),
        traffic_df[['from_stop_seq', 'to_stop_seq', 'direction', 'timestamp', 'congestion_ratio']]
                  .sort_values('timestamp'),
        left_on='arrival_timestamp',
        right_on='timestamp',
        by=['from_stop_seq', 'to_stop_seq', 'direction'],
        direction='nearest',
        tolerance=pd.Timedelta('30min')
    ).drop(columns=['timestamp', 'direction'], errors='ignore')

    filled = df['congestion_ratio'].isna().sum()
    df['congestion_ratio'] = df['congestion_ratio'].fillna(1.0)
    print(f'Eşleşen: {len(df)-filled}/{len(df)}, {filled} satır 1.0 ile dolduruldu')
else:
    print('Trafik verisi henüz yok — congestion_ratio=1.0 atandı')
    df['congestion_ratio'] = 1.0

## Adım 7 — Mekansal Featurelar

**Düzeltmeler:**
- `iterrows()` → vektörize Haversine
- `stop_progress` formülü: `(max_seq - to_seq) / (max_seq - 1)` → son durağa varışta 1.0

In [ ]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return (R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))).round(1)


# Durak koordinatlarını DataFrame'e çevir, merge ile ekle (join = vektörize)
coord_rows = (
    [{'yon': 0, 'seq': s['seq'], 'lat': s['lat'], 'lon': s['lon']} for s in STOPS_DIR0] +
    [{'yon': 1, 'seq': s['seq'], 'lat': s['lat'], 'lon': s['lon']} for s in STOPS_DIR1]
)
coord_df = pd.DataFrame(coord_rows)

df = df.merge(
    coord_df.rename(columns={'seq': 'from_stop_seq', 'lat': 'from_lat', 'lon': 'from_lon'}),
    on=['yon', 'from_stop_seq'], how='left'
).merge(
    coord_df.rename(columns={'seq': 'to_stop_seq', 'lat': 'to_lat', 'lon': 'to_lon'}),
    on=['yon', 'to_stop_seq'], how='left'
)

df['distance_m'] = haversine_vectorized(
    df['from_lat'], df['from_lon'], df['to_lat'], df['to_lon']
)

# stop_progress: to_seq=1 (son durak) → 1.0, to_seq=max_seq-1 (ilk segment) → küçük değer
max_seq = {0: max(s['seq'] for s in STOPS_DIR0),
           1: max(s['seq'] for s in STOPS_DIR1)}

df['stop_progress'] = df.apply(
    lambda r: round((max_seq[int(r['yon'])] - int(r['to_stop_seq'])) / (max_seq[int(r['yon'])] - 1), 3),
    axis=1
)

df = df.drop(columns=['from_lat', 'from_lon', 'to_lat', 'to_lon'])

print(f'distance_m: ort={df["distance_m"].mean():.0f}m, std={df["distance_m"].std():.0f}m, NaN={df["distance_m"].isna().sum()}')
print(f'stop_progress: min={df["stop_progress"].min()}, max={df["stop_progress"].max()}')

## Adım 8 — Final Dataset

**Feature özeti:**

| Grup | Featurelar |
|------|-----------|
| Zamansal | `hour`, `day_of_week`, `day_type`, `time_block`, `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos` |
| Mekansal | `stop_progress`, `distance_m` |
| Sequential (lag) | `prev_travel_time_min`, `prev_deviation` |
| GTFS | `scheduled_travel_min` |
| Hava | `temperature`, `humidity`, `precipitation`, `wind_speed`, `visibility`, `weather_cat_enc` |
| Trafik | `congestion_ratio` |

In [ ]:
ID_COLS = [
    'date', 'bus_id', 'route_id', 'yon', 'trip_start_time',
    'from_stop_seq', 'to_stop_seq', 'from_stop_name', 'to_stop_name',
    'arrival_timestamp', 'travel_seconds',
]

FEATURE_COLS = [
    # Zamansal
    'hour', 'day_of_week', 'day_type', 'time_block',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    # Mekansal
    'stop_progress', 'distance_m',
    # Sequential lag
    'prev_travel_time_min', 'prev_deviation',
    # GTFS
    'scheduled_travel_min',
    # Hava
    'temperature', 'humidity', 'precipitation', 'wind_speed', 'visibility', 'weather_cat_enc',
    # Trafik
    'congestion_ratio',
]

TARGET_COL = 'travel_time_min'

avail_id   = [c for c in ID_COLS      if c in df.columns]
avail_feat = [c for c in FEATURE_COLS if c in df.columns]
df_final   = df[avail_id + avail_feat + [TARGET_COL]].copy()

print(f'Final dataset: {df_final.shape[0]} satır, {len(avail_feat)} feature')
print(f'Eksik feature: {[c for c in FEATURE_COLS if c not in df.columns]}')
print(f'\nNaN özeti:')
nan_s = df_final[avail_feat].isna().sum()
print(nan_s[nan_s > 0].to_string() if nan_s.sum() > 0 else 'Yok')
print(f'\nHedef (travel_time_min):')
print(df_final[TARGET_COL].describe().round(3))

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_final.to_csv(OUTPUT_CSV, index=False)
print(f'Kaydedildi: {OUTPUT_CSV}')
print(f'Boyut: {os.path.getsize(OUTPUT_CSV)/1024:.1f} KB — {len(df_final)} satır, {len(df_final.columns)} kolon')

## Ek — Görselleştirme

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('matplotlib kurulu değil'); raise SystemExit

if len(df_final) < 10:
    print('Görselleştirme için yetersiz veri.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    axes[0,0].hist(df_final[TARGET_COL], bins=30, edgecolor='black', alpha=0.7)
    axes[0,0].set_xlabel('Seyahat süresi (dk)')
    axes[0,0].set_title('Hedef dağılımı')

    axes[0,1].scatter(df_final['scheduled_travel_min'], df_final[TARGET_COL], alpha=0.4, s=10)
    lim = max(df_final['scheduled_travel_min'].max(), df_final[TARGET_COL].max()) + 0.5
    axes[0,1].plot([0, lim], [0, lim], 'r--', alpha=0.6)
    axes[0,1].set_xlabel('Planlanmış (dk)'); axes[0,1].set_ylabel('Gerçek (dk)')
    axes[0,1].set_title('Gerçek vs GTFS')

    axes[0,2].scatter(df_final['prev_travel_time_min'], df_final[TARGET_COL], alpha=0.3, s=8)
    axes[0,2].set_xlabel('Önceki segment süresi (dk)'); axes[0,2].set_ylabel('Şimdiki (dk)')
    axes[0,2].set_title('Lag feature korelasyonu')

    by_hour = df_final.groupby('hour')[TARGET_COL].mean()
    axes[1,0].bar(by_hour.index, by_hour.values, alpha=0.7, color='steelblue')
    axes[1,0].set_xlabel('Saat'); axes[1,0].set_ylabel('Ort. süre (dk)')
    axes[1,0].set_title('Saate göre ortalama süre')

    axes[1,1].hist(df_final['congestion_ratio'], bins=20, edgecolor='black', alpha=0.7, color='tomato')
    axes[1,1].set_xlabel('Congestion ratio')
    axes[1,1].set_title('Trafik yoğunluğu dağılımı')

    by_seq = df_final.groupby('to_stop_seq')[TARGET_COL].mean().sort_index()
    axes[1,2].bar(by_seq.index.astype(str), by_seq.values, alpha=0.7, color='green')
    axes[1,2].set_xlabel('to_stop_seq'); axes[1,2].set_ylabel('Ort. süre (dk)')
    axes[1,2].set_title('Durak bazında ortalama süre')
    axes[1,2].tick_params(axis='x', rotation=60, labelsize=6)

    plt.tight_layout()
    fig_path = os.path.join(PROJECT_ROOT, 'results', 'figures', 'feature_engineering_v2_eda.png')
    os.makedirs(os.path.dirname(fig_path), exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Kaydedildi: {fig_path}')